# 04 - Baseline Models

In [1]:
import os
import pandas as pd
from src.utils import FEATURES_DATA_PATH, TARGET_COL
from src.forecasting import chronological_split, naive_forecast, moving_average_forecast
from src.evaluation import evaluate, build_comparison_table

df = pd.read_csv(FEATURES_DATA_PATH, parse_dates=['date'])
train, val, test, meta = chronological_split(df)
print(f"Train: {len(train)} rows (up to {meta['train_end'].date()})")
print(f"Val:   {len(val)} rows (up to {meta['val_end'].date()})")
print(f"Test:  {len(test)} rows (after {meta['val_end'].date()})")

Train: 49200 rows (up to 2023-06-04)
Val:   10500 rows (up to 2023-09-17)
Test:  10600 rows (after 2023-09-17)


## Baseline 1: Naive forecast

In [2]:
results = {}
results['Naive (lag_1)'] = evaluate(test[TARGET_COL], naive_forecast(test))
results['Naive (lag_1)']

{'MAE': 119.589, 'RMSE': 152.944, 'MAPE_%': 361.918, 'WAPE_%': 87.606}

## Baseline 2: Moving-average forecast

In [3]:
results['Moving Average (7d)'] = evaluate(test[TARGET_COL], moving_average_forecast(test, 7))
results['Moving Average (7d)']

{'MAE': 93.319, 'RMSE': 115.519, 'MAPE_%': 347.682, 'WAPE_%': 68.361}

In [4]:
build_comparison_table(results)

,MAE,RMSE,MAPE_%,WAPE_%
Naive (lag_1),119.589,152.944,361.918,87.606
Moving Average (7d),93.319,115.519,347.682,68.361


In [9]:
import numpy as np
from src.utils import PROCESSED_DATA_PATH
from src.forecasting import fit_sarima_all_series

raw = pd.read_csv(PROCESSED_DATA_PATH, parse_dates=['date'])
raw = raw.sort_values(['store_id','product_id','date']).reset_index(drop=True)

pred_series = fit_sarima_all_series(raw, TARGET_COL)
raw['holt_pred'] = pred_series.clip(lower=0)
print('Used model: SARIMA')

dates = np.sort(raw['date'].unique())
n = len(dates)
val_end = dates[int(n*0.85)-1]
classical_test = raw[raw['date'] > val_end]
print('test rows:', len(classical_test))

d:\Projects\Retail_Store_Inventory\venv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Projects\Retail_Store_Inventory\venv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Projects\Retail_Store_Inventory\venv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Projects\Retail_Store_Inventory\venv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueW

Used model: SARIMA
test rows: 11000


In [6]:
results["SARIMA (classical)"] = evaluate(classical_test[TARGET_COL], classical_test['holt_pred'])
results["SARIMA (classical)"]

{'MAE': 88.743, 'RMSE': 108.756, 'MAPE_%': 337.82, 'WAPE_%': 65.044}

In [7]:
build_comparison_table(results)

,MAE,RMSE,MAPE_%,WAPE_%
Naive (lag_1),119.589,152.944,361.918,87.606
Moving Average (7d),93.319,115.519,347.682,68.361
SARIMA (classical),88.743,108.756,337.820,65.044


In [8]:
import json

with open("../data/processed/baseline_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved baseline metrics:", results)

Saved baseline metrics: {'Naive (lag_1)': {'MAE': 119.589, 'RMSE': 152.944, 'MAPE_%': 361.918, 'WAPE_%': 87.606}, 'Moving Average (7d)': {'MAE': 93.319, 'RMSE': 115.519, 'MAPE_%': 347.682, 'WAPE_%': 68.361}, 'SARIMA (classical)': {'MAE': 88.743, 'RMSE': 108.756, 'MAPE_%': 337.82, 'WAPE_%': 65.044}}
